# B. A 1-D CNN finds a DNA motif, in PyTorch

The PyTorch counterpart of GlassNN example `06_cnn_dna_motif` (lecture 11,
book chapter 9). Run it on [Google Colab](https://colab.research.google.com),
where PyTorch is preinstalled.

**Goals**

- Train the same network, `Conv1d` → ReLU → global max pooling → `Linear`,
  on the same data, with `torch.utils.data.DataLoader`.
- Read the learned filter as a position weight matrix.

**Expected finding** (the same as in GlassNN): more than 95 % test accuracy,
and a filter whose consensus contains most of the motif `TGACGTCA`.

The data generator is the file `dna_motifs.py` of the GlassNN examples.
The next cell downloads it from GitHub, at the GlassNN version used by the
course, unless it is already here.

In [ ]:
import pathlib
import urllib.request

GLASSNN_TAG = "v0.1.2"
if not pathlib.Path("dna_motifs.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bioinfo-recetox/glassnn/"
        f"{GLASSNN_TAG}/examples/dna_motifs.py",
        "dna_motifs.py",
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from dna_motifs import make_motif_dataset, one_hot
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

MOTIF = "TGACGTCA"
sequences, labels, positions = make_motif_dataset(
    2000, motif=MOTIF, mutation_rate=0.0, seed=1
)
X = torch.from_numpy(one_hot(sequences)).float()  # (N, 4, L)
y = torch.from_numpy(np.asarray(labels)).long()
X_train, y_train, X_test, y_test = X[:1000], y[:1000], X[1000:], y[1000:]
print(sequences[0], "label", labels[0], "| one-hot shape:", tuple(X.shape))

In GlassNN, `DataLoader(X, y, ...)` takes the arrays; in PyTorch they are
wrapped in a `TensorDataset` first. The training loop is the same.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Conv1d(4, 16, kernel_size=8),
    nn.ReLU(),
    nn.MaxPool1d(X.shape[2] - 8 + 1),  # global max pooling
    nn.Flatten(),
    nn.Linear(16, 2),
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
for _ in range(30):
    for xb, yb in loader:
        optimizer.zero_grad()
        nn.functional.cross_entropy(model(xb), yb).backward()
        optimizer.step()

with torch.no_grad():
    accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
print(model)
print("test accuracy:", accuracy)

In [ ]:
votes = model[4].weight[1] - model[4].weight[0]
best = int(votes.argmax())
W = model[0].weight[best].detach().numpy()  # (4, 8)
consensus = "".join("ACGT"[i] for i in W.argmax(axis=0))
print("filter", best, "consensus:", consensus, "| motif:", MOTIF)

fig, ax = plt.subplots(figsize=(6, 2.5))
image = ax.imshow(W, cmap="RdBu_r", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
ax.set_yticks(range(4), list("ACGT"))
ax.set_xlabel("position in the filter")
fig.colorbar(image)
plt.show()

**Finding.** The accuracy and the learned filter match the GlassNN
notebook; the filter may be shifted by a position or two with respect to
the motif. Note `.detach().numpy()`: a PyTorch parameter must leave the
graph before NumPy can read it, while GlassNN's `.data` is already a NumPy
array.

## Rosetta table: GlassNN and PyTorch

Names, arguments and behaviour that differ between the two libraries (the same table as in chapter 12 of the GlassNN book). Everything not listed has the same name and meaning.

| Area | GlassNN | PyTorch | Difference |
|---|---|---|---|
| Tensors | `Tensor` | `torch.Tensor` | A float64 NumPy array is cast to the default dtype (PyTorch keeps float64). The graph is kept after `backward()`, so a second call works without `retain_graph=True`. There are no in-place operations (`+=`, item assignment); they raise an error. |
| Tensors | `Tensor.numpy` | `torch.Tensor.numpy()` | Returns a copy; PyTorch shares memory with the tensor. |
| Tensors | `backend.asarray` | `torch.as_tensor` | Floating-point input becomes the default dtype, so float64 data never mixes silently with float32 parameters. |
| Device and randomness | `backend.set_backend` | `tensor.to("cuda")`, `model.to("cuda")` | One global switch: all tensors created afterwards are on the GPU (CuPy). PyTorch places each tensor on a device. |
| Device and randomness | `backend.manual_seed` | `torch.manual_seed` | Returns a `numpy.random.Generator`, the same on CPU and GPU, so a seed gives identical runs on both; it is unseeded until called. PyTorch returns a `torch.Generator`, starts from a fixed seed, and draws different numbers on each device. |
| Device and randomness | `backend.from_dlpack`, `backend.to_dlpack` | `torch.from_dlpack`, `torch.utils.dlpack.to_dlpack` | Work on arrays, not tensors: wrap the result with `Tensor(...)`, pass `tensor.data`. |
| Data | `data.DataLoader` | `torch.utils.data.DataLoader` | Takes the arrays `X, y` directly instead of a `Dataset`; no workers, samplers or `collate_fn`. |
| Data | `data.one_hot` | `torch.nn.functional.one_hot` | Lives in `glassnn.data`. |
| Modules | `nn.Module.state_dict` | `Module.state_dict()` | Returns copies of the values, not tensors that share memory; parameters and buffers are listed module by module. |
| Modules | `nn.Module.register_buffer` | `Module.register_buffer(..., persistent=True)` | No `persistent` argument: every buffer is saved. |
| Layers | `nn.Linear` | `nn.Linear` | Extra argument `parametrization="ntk"` (NTK scaling, chapter 8); the default initialization is PyTorch's. |
| Layers | `nn.Conv1d`, `nn.Conv2d` | `nn.Conv1d`, `nn.Conv2d` | No `groups` and no `padding_mode` (zero padding only). |
| Layers | `functional.conv1d`, `functional.conv2d` | `F.conv1d`, `F.conv2d` | No `groups`; zero padding only. |
| Layers | `functional.max_pool1d`, `functional.max_pool2d`, `nn.MaxPool1d`, `nn.MaxPool2d` | `F.max_pool1d`, `nn.MaxPool1d`, ... | No `dilation`, `ceil_mode` or `return_indices`. |
| Layers | `functional.avg_pool1d`, `functional.avg_pool2d`, `nn.AvgPool1d`, `nn.AvgPool2d` | `F.avg_pool1d`, `nn.AvgPool1d`, ... | No `ceil_mode`; padding always counts in the mean (`count_include_pad=True`). |
| Layers | `nn.BatchNorm1d` | `nn.BatchNorm1d` | No `num_batches_tracked` buffer and no `momentum=None` (cumulative average). |
| Layers | `nn.Dropout` | `nn.Dropout` | No `inplace` argument. |
| Layers | `functional.dropout` | `F.dropout` | No `inplace` argument; an extra `generator=` argument. |
| Attention | `functional.scaled_dot_product_attention` | `F.scaled_dot_product_attention` | A boolean mask is `True` where attention is **forbidden** (PyTorch, in this function only: `True` = allowed). |
| Attention | `nn.MultiheadAttention` | `nn.MultiheadAttention` | Batch-first only, `(N, L, E)`. Three projections `q_proj`, `k_proj`, `v_proj` instead of one packed `in_proj_weight`, so `state_dict` names differ. No `kdim`, `vdim`, `add_bias_kv`, `add_zero_attn`. `is_causal=True` without a mask builds the causal mask. |
| Attention | `nn.TransformerEncoderLayer` | `nn.TransformerEncoderLayer` | Batch-first only. Defaults `norm_first=True` and `activation="gelu"` (PyTorch: `False`, `"relu"`); `activation` is a string. |
| Attention | `nn.TransformerEncoder` | `nn.TransformerEncoder` | No `enable_nested_tensor` or `mask_check`. |
| Attention | `nn.PositionalEncoding` | none (defined in PyTorch tutorials) | A GlassNN module: sinusoidal, batch-first, dropout 0 by default. |
| Losses | `functional.cross_entropy`, `nn.CrossEntropyLoss` | `F.cross_entropy`, `nn.CrossEntropyLoss` | Only `(N, C)` logits with integer targets: no class probabilities, `weight` or `ignore_index`. |
| Losses | `functional.mse_loss` | `F.mse_loss` | Different shapes raise an error (PyTorch broadcasts with a warning); `binary_cross_entropy_with_logits` does the same. |
| Optimizers | `optim.Optimizer` | `torch.optim.Optimizer` | One parameter group; the rate is `optimizer.lr` (PyTorch: `optimizer.param_groups[0]["lr"]`). |
| Optimizers | `optim.LinearWarmup` | `lr_scheduler.LinearLR(optimizer, start_factor, end_factor=1.0, total_iters=warmup_steps)` | The same schedule under a different name; all schedulers are also in `glassnn.optim`. |
| Optimizers | `optim.clip_grad_norm_` | `torch.nn.utils.clip_grad_norm_` | Lives in `glassnn.optim`; Euclidean norm only. |
| Tools | `gradcheck.gradcheck` | `torch.autograd.gradcheck` | Checks one vector-Jacobian product with fixed weights instead of the full Jacobian, and raises an error that names the element and both values. |
| Tools | `analysis.empirical_ntk`, `analysis.activation_stats`, `analysis.count_parameters` | none (`torch.func.jacrev`, forward hooks, `sum(p.numel() ...)`) | GlassNN helpers for chapters 4 and 8; the Colab notebooks show the PyTorch equivalents. |